[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [asyncpg and psycopg3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)

# Pipeline Mode


## What you will be able to do

Send many statements without waiting for each answer, and say what that saves and what it does not.
Check that the libpq in front of you can do it at all, rather than assuming. Read the results back
afterwards, and recognize the error that arrives at a statement which was perfectly fine. Know the
three things that cannot be done inside a pipeline, and why each one is excluded. And say why the
measurement in this notebook understates the feature.


## The idea

### The problem

Every statement so far has been a conversation: send, wait, read, send the next. On a local socket
the waiting is almost nothing. Over a network with a millisecond of latency, five thousand statements
is five seconds of a program doing nothing at all.

Pipeline mode removes the waiting. The statements go out one after another without stopping for
answers, and the answers are collected afterwards. What it does not remove is the work: the server
still parses, plans and runs every one of them.

### What a pipeline is

A mode on the connection, entered with `with conn.pipeline():`. Inside it, `execute` queues a
statement and returns immediately. The results arrive when you ask for them, or when the block ends,
whichever is first.

That is a real change in what `execute` means, and it is why errors behave so strangely: at the
moment a statement fails, your code has moved on, and there is no way to raise it at the line that
caused it.

### Why it works that way

The protocol allows a client to send several messages before reading replies, which is what libpq's
pipeline mode exposes. It needs libpq 14 or newer, and `psycopg[binary]` bundles its own, so this
works regardless of how old the PostgreSQL server or the system libpq is.

The server processes the queue in order and, if one statement fails, refuses the rest of the group
until a synchronization point. That is what makes a pipeline all-or-nothing in a way a series of
statements is not.

### Where this shows up

Any loop of small statements against a database that is not on the same machine. It is also already
working for you in one place: `executemany` in **Placeholders and Identifiers** uses pipeline mode
internally, which is why it is not one round trip per row.

### What this notebook covers

Checking for the capability. Queueing statements and reading the results. The error that names the
wrong statement. The three things that are not allowed inside a pipeline. What it is worth, measured
honestly. Then the four failures.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import psycopg

with psycopg.connect("dbname=guide", autocommit=True) as conn:
    with conn.pipeline():
        sent = [(sql, conn.cursor()) for sql in ("SELECT 1", "SELECT 1/0", "SELECT 2")]
        for sql, cur in sent:
            cur.execute(sql)                        # none of these has been answered yet

        for sql, cur in sent:
            try:
                print(f"  {sql:<12} -> {cur.fetchone()}")
            except Exception as error:
                print(f"  {sql:<12} -> {type(error).__name__}: {error}")
```

```
  SELECT 1     -> DivisionByZero: division by zero
  SELECT 1/0   -> ProgrammingError: no result available
  SELECT 2     -> ProgrammingError: no result available
```

Read those three lines again. `SELECT 1` cannot divide by zero, and it is the one that raised. The
statement that really failed reports that it has no result, and so does the one after it. That is
the cost of not waiting: by the time the failure came back, the code had moved on, and the only place
left to report it was the next thing anybody asked about.


## Setup

Eight imports, psycopg, the server, and a table to write into.

- `psycopg` is the driver here, and `errors` is the exception classes. asyncpg has no pipeline mode
  of this kind, so this notebook is psycopg only
- `time` measures, and `subprocess`, `sys`, `os`, `getpass` stand the server up with `version` and
  `PackageNotFoundError`

Setup prints whether pipeline mode is available rather than assuming it. It needs libpq 14 or newer,
and `psycopg[binary]` brings its own libpq, so the answer does not depend on the server's version or
on anything installed by the system.


In [1]:
import getpass
import os
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("psycopg") < "3.3" or version("asyncpg") < "0.31":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "psycopg[binary,pool]==3.3.6", "psycopg-pool==3.3.2", "asyncpg==0.31.0"],
                   check=True)

import psycopg
from psycopg import errors

def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(database="postgres"):
    """Whether a server is there, asked the only way that needs no client binaries."""
    try:
        with psycopg.connect(f"dbname={database}", connect_timeout=2):
            return True
    except psycopg.OperationalError:
        return False


def start_server(wait=60):
    """Install and start PostgreSQL if nothing is answering. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No PostgreSQL is answering. Start your own server and run this again: "
                           "this cell only installs one on Linux, which is what Colab runs.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"{sudo}apt-get -qq update")
    shell(f"{sudo}apt-get -qq -y install postgresql postgresql-contrib")
    shell(f"{sudo}service postgresql start")                        # Colab has no systemd

    for attempt in range(1, wait + 1):                              # start returns before it listens
        if shell("pg_isready -q")[0] == 0:
            break
        print(f"  waiting for the cluster ({attempt})")              # a silent minute looks hung
        time.sleep(1)
    else:
        raise RuntimeError(f"PostgreSQL did not accept connections within {wait} seconds.")

    me = getpass.getuser()                                          # peer authentication wants a role
    asking = f"""sudo -u postgres psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='{me}'" """
    if shell(asking)[1] != "1":                                     # named for the operating system user
        shell(f"sudo -u postgres createuser -s {me}")
    return "installed and started"

def build(rows=5000):
    """Make the guide database and its events table, and fill it once."""
    with psycopg.connect("dbname=postgres", autocommit=True) as conn:
        if not conn.execute("SELECT 1 FROM pg_database WHERE datname = 'guide'").fetchone():
            conn.execute("CREATE DATABASE guide")                   # cannot run in a transaction

    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        for (leftover,) in conn.execute(                            # whatever an earlier run made
                "SELECT tablename FROM pg_tables "
                "WHERE schemaname = 'public' AND tablename <> 'events'").fetchall():
            conn.execute(f'DROP TABLE IF EXISTS "{leftover}" CASCADE')

        conn.execute("""CREATE TABLE IF NOT EXISTS events (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        if conn.execute("SELECT count(*) FROM events").fetchone()[0] == 0:
            conn.execute("""INSERT INTO events (kind, payload)
                            SELECT (ARRAY['click', 'view', 'purchase'])[1 + n %% 3],
                                   jsonb_build_object('n', n, 'size', 1 + n %% 7)
                            FROM generate_series(1, %s) AS n""", (rows,))
        return conn.execute("SELECT count(*) FROM events").fetchone()[0]

def report():
    """One line naming what this notebook is running against."""
    rows = build()                                                  # makes the database if it is new
    with psycopg.connect("dbname=guide") as conn:
        major = int(conn.execute("SHOW server_version_num").fetchone()[0]) // 10000
    return (f"PostgreSQL {major} | psycopg {version('psycopg')} | asyncpg {version('asyncpg')} "
            f"| events: {rows} rows")

STATEMENTS = 5000


def fresh():
    """An empty table, so each timing starts from the same place."""
    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        conn.execute("DROP TABLE IF EXISTS piped")
        conn.execute("CREATE TABLE piped (id int PRIMARY KEY, note text)")


def rows():
    with psycopg.connect("dbname=guide") as conn:
        return conn.execute("SELECT count(*) FROM piped").fetchone()[0]


def timed(work):
    fresh()
    start = time.perf_counter()
    work()
    return time.perf_counter() - start


def against(baseline, measured):
    """How much faster, as a band, because a timing is not repeatable to a digit."""
    ratio = baseline / measured
    if ratio < 1.2:
        return "no faster"
    if ratio < 3:
        return "somewhat faster"
    return "much faster"


print("server:", start_server())
print(report())
print("pipeline mode available:", psycopg.capabilities.has_pipeline())
fresh()


server: already running
PostgreSQL 16 | psycopg 3.3.6 | asyncpg 0.31.0 | events: 5000 rows
pipeline mode available: True


## Worked examples

### Queueing, and collecting

Inside a pipeline, `execute` returns before the server has answered. The cursor holds a place in the
queue:


In [2]:
fresh()

with psycopg.connect("dbname=guide") as conn:
    with conn.pipeline():
        for number in range(5):
            conn.execute("INSERT INTO piped VALUES (%s, %s)", (number, f"row {number}"))
        print("inside the block, statements queued: 5")

print("after the block, rows in the table:", rows())


inside the block, statements queued: 5
after the block, rows in the table: 5


The rows are there afterwards, and nothing waited for anything in between. Leaving the block is what
makes sure every statement has been sent and every answer read.

Reading results needs a cursor per statement, because one cursor can only hold one result:


In [3]:
with psycopg.connect("dbname=guide") as conn:
    with conn.pipeline():
        cursors = [conn.cursor() for _ in range(3)]
        for number, cur in enumerate(cursors):
            cur.execute("SELECT %s * 10", (number + 1,))

        print("answers:", [cur.fetchone()[0] for cur in cursors])


answers: [10, 20, 30]


That is the shape to remember: queue with one cursor each, then read. Reusing a single cursor inside
a pipeline overwrites its result and is the second of the Common errors.

### What a sync does

`sync` is a boundary. It waits for everything queued so far, and it is what separates one group of
statements from the next:


In [4]:
fresh()

with psycopg.connect("dbname=guide", autocommit=True) as conn:
    with conn.pipeline() as pipeline:
        conn.execute("INSERT INTO piped VALUES (1, 'before the sync')")
        pipeline.sync()
        print("after the first sync, rows:", rows())

        conn.execute("INSERT INTO piped VALUES (2, 'after it')")
        pipeline.sync()
        print("after the second sync, rows:", rows())


after the first sync, rows: 1
after the second sync, rows: 2


Without the syncs those counts would both have been whatever the queue happened to have flushed.
With them, each group is finished before the next begins, which is also what limits the damage when
one statement in a group fails.

### The three things a pipeline will not do

Each of them is excluded for a reason to do with the queue:


In [5]:
def attempt(what, work):
    with psycopg.connect("dbname=guide") as conn:
        try:
            work(conn)
            print(f"  {what:<22} allowed")
        except (psycopg.NotSupportedError, psycopg.ProgrammingError) as error:
            print(f"  {what:<22} {type(error).__module__}.{type(error).__name__}: {error}")


def with_copy(conn):
    with conn.pipeline():
        with conn.cursor() as cur:
            with cur.copy("COPY piped FROM STDIN") as copy:
                copy.write_row((9, "x"))


def with_named_cursor(conn):
    with conn.pipeline():
        with conn.cursor(name="c") as cur:
            cur.execute("SELECT 1")


def with_stream(conn):
    with conn.pipeline():
        with conn.cursor() as cur:
            list(cur.stream("SELECT 1"))


attempt("COPY", with_copy)
attempt("a named cursor", with_named_cursor)
attempt("stream()", with_stream)


  COPY                   psycopg.NotSupportedError: COPY cannot be used in pipeline mode
  a named cursor         psycopg.NotSupportedError: server-side cursors not supported in pipeline mode
  stream()               psycopg.ProgrammingError: stream() cannot be used in pipeline mode


All three need the connection to itself. `COPY` takes it over for the length of the load, a named
cursor is a conversation of its own, and `stream` puts the connection into single-row mode, which
**Server-Side Cursors** showed is a state the connection cannot be interrupted in.

Note the module names in those messages: two say `psycopg.NotSupportedError` rather than
`psycopg.errors.NotSupportedError`, because the DB-API base classes live in the top-level module.

### What it is worth

Five thousand single-row statements, with and without a pipeline:


In [6]:
def one_at_a_time():
    with psycopg.connect("dbname=guide") as conn:
        for number in range(STATEMENTS):
            conn.execute("INSERT INTO piped VALUES (%s, %s)", (number, "x"))


def in_a_pipeline():
    with psycopg.connect("dbname=guide") as conn:
        with conn.pipeline():
            for number in range(STATEMENTS):
                conn.execute("INSERT INTO piped VALUES (%s, %s)", (number, "x"))


baseline = timed(one_at_a_time)
print(f"{STATEMENTS} statements, over a local socket")
print("  one at a time:", "the baseline")
print("  in a pipeline:", against(baseline, timed(in_a_pipeline)))


5000 statements, over a local socket
  one at a time: the baseline
  in a pipeline: somewhat faster


That is an honest and unimpressive number, and the reason matters more than the number does. A Unix
socket on the same machine is the smallest round trip there is, so the waiting a pipeline removes is
nearly free here, and almost all of what is left is the server doing the work.

The arithmetic for a real network is worth doing in your head. At one millisecond of latency, five
thousand statements spend five seconds waiting. A pipeline removes essentially all of that, and this
measurement cannot show it because there is no latency here to remove.

`COPY` remains the answer for bulk loading, as **COPY** showed: it removes the per-statement work as
well as the waiting.

### When to reach for which

| What you are doing | What to use |
|---|---|
| one statement | `execute` |
| the same statement, many rows | `executemany`, which pipelines already |
| bulk loading | `COPY` |
| many different statements, over a network | a pipeline |
| many different statements, locally | plain `execute`, and do not bother |
| a group that must finish before the next | `pipeline.sync()` between them |
| reading results from a pipeline | one cursor per statement |

The default is not to use a pipeline. Reach for it when a program is making many small round trips
to a database that is not on the same machine, and measure before and after rather than assuming.

### A batch of small statements, finished

Everything above, as the shape a real job would use: a group per sync, one cursor per statement, and
the results collected with each statement's own identity attached.


In [7]:
def apply_updates(updates, group=100):
    """Run many small statements in groups, and report what each one did."""
    done = []
    fresh()
    with psycopg.connect("dbname=guide") as conn:
        with conn.pipeline() as pipeline:
            for start in range(0, len(updates), group):
                batch = updates[start:start + group]
                cursors = []
                for identifier, note in batch:
                    cur = conn.cursor()
                    cur.execute("INSERT INTO piped VALUES (%s, %s)", (identifier, note))
                    cursors.append((identifier, cur))
                pipeline.sync()                                     # this group is finished
                done.extend((identifier, cur.rowcount) for identifier, cur in cursors)
    return done


results = apply_updates([(n, f"note {n}") for n in range(250)])
print("statements run:", len(results))
print("first three:   ", results[:3])
print("rows in the table:", rows())


statements run: 250
first three:    [(0, 1), (1, 1), (2, 1)]
rows in the table: 250


The `sync` at the end of each group is what makes `rowcount` readable for that group, and it is what
keeps a failure from taking the whole job down rather than one group of a hundred.

### Where each part came from

| In the batch | What it relies on | The section that showed it |
|---|---|---|
| `with conn.pipeline()` | statements queued rather than awaited | Queueing, and collecting |
| a cursor per statement | one result per cursor | Queueing, and collecting |
| `pipeline.sync()` per group | a boundary that finishes what came before | What a sync does |
| `cur.rowcount` after the sync | a result that has actually arrived | What a sync does |
| `%s` for the values | a value is a value | **Placeholders and Identifiers** |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/09-pipeline-mode-solutions.ipynb).

**1.** Check whether pipeline mode is available, and print the answer.


In [8]:
# your code here


**2.** Queue five inserts in a pipeline and count the rows after the block ends.


In [9]:
# your code here


**3.** Queue three different `SELECT` statements with a cursor each and print all three answers.


In [10]:
# your code here


**4.** Put a failing statement in the middle of three and print what each cursor gives back.


In [11]:
# your code here


**5.** Show that `COPY` is refused inside a pipeline, and print the class of the error.


In [12]:
# your code here


**6.** Time a thousand single-row inserts with and without a pipeline, and print which band the
difference falls in.


In [13]:
# your code here


## Common errors

### psycopg.errors.DivisionByZero, raised by a statement that cannot divide


In [14]:
with psycopg.connect("dbname=guide", autocommit=True) as conn:
    with conn.pipeline():
        innocent = conn.cursor()
        guilty = conn.cursor()
        innocent.execute("SELECT 1")
        guilty.execute("SELECT 1/0")

        innocent.fetchone()


DivisionByZero: division by zero

`SELECT 1` raised a division error. Nothing is wrong with psycopg: the failure came back from the
server while the code was already past both statements, and the first `fetchone` is the first
opportunity anybody gave it to report anything.

This is the thing to internalize about pipelines. A traceback inside one points at where you were
standing, not at what went wrong. The statement that failed is found by reading the results in
order, which is why a cursor per statement is worth the bookkeeping:


In [15]:
with psycopg.connect("dbname=guide", autocommit=True) as conn:
    sent = []
    with conn.pipeline():
        for sql in ("SELECT 1", "SELECT 1/0", "SELECT 2"):
            cur = conn.cursor()
            cur.execute(sql)
            sent.append((sql, cur))

        for sql, cur in sent:
            try:
                print(f"  {sql:<12} ok:  {cur.fetchone()}")
            except psycopg.Error as error:
                print(f"  {sql:<12} bad: {type(error).__name__}: {error}")


  SELECT 1     bad: DivisionByZero: division by zero
  SELECT 1/0   bad: ProgrammingError: no result available
  SELECT 2     bad: ProgrammingError: no result available


### No error, and the wrong answer: one cursor for two statements


In [16]:
with psycopg.connect("dbname=guide", autocommit=True) as conn:
    with conn.pipeline():
        cur = conn.cursor()
        cur.execute("SELECT 1")
        cur.execute("SELECT 2")                                     # the same cursor, twice

        print("asking for the first statement's answer:", cur.fetchone())
        print("asking again:                           ", cur.fetchone())


asking for the first statement's answer: (2,)
asking again:                            None


Nothing raised, and the answer to `SELECT 1` is gone. A cursor holds one result, so the second
`execute` took the place of the first, and the value that came back for it belongs to the other
statement entirely.

Outside a pipeline this cannot happen, because `execute` does not return until its result has
arrived, so reusing a cursor is the ordinary thing to do. Inside one, both statements are in flight
and the cursor can only keep the last. A cursor each is the answer, and it costs nothing:


In [17]:
with psycopg.connect("dbname=guide", autocommit=True) as conn:
    with conn.pipeline():
        first, second = conn.cursor(), conn.cursor()
        first.execute("SELECT 1")
        second.execute("SELECT 2")
        print("both:", first.fetchone(), second.fetchone())


both: (1,) (2,)


### psycopg.NotSupportedError: COPY cannot be used in pipeline mode


In [18]:
with psycopg.connect("dbname=guide") as conn:
    with conn.pipeline():
        with conn.cursor() as cur:
            with cur.copy("COPY piped FROM STDIN") as copy:
                copy.write_row((1, "x"))


NotSupportedError: COPY cannot be used in pipeline mode

A `COPY` needs the connection to itself while rows stream across it, and a pipeline is a queue of
statements sharing that connection. The two cannot both be true.

The same applies to a named cursor and to `stream`, with their own messages. None of it is a
limitation worth fighting: `COPY` is already faster than anything a pipeline could do for a bulk
load, so the answer is to do it outside:


In [19]:
fresh()
with psycopg.connect("dbname=guide") as conn:
    with conn.pipeline():                                           # the small statements
        for number in range(5):
            conn.execute("INSERT INTO piped VALUES (%s, %s)", (number, "from the pipeline"))

    with conn.cursor() as cur:                                      # and the bulk load, outside it
        with cur.copy("COPY piped (id, note) FROM STDIN") as copy:
            for number in range(5, 10):
                copy.write_row((number, "from the copy"))

print("rows:", rows())


rows: 10


### No error, and a pipeline that bought nothing: a local socket


In [20]:
baseline = timed(one_at_a_time)
print(f"{STATEMENTS} statements:")
print("  one at a time:", "the baseline")
print("  in a pipeline:", against(baseline, timed(in_a_pipeline)))
print()
print("what a pipeline removes is the waiting, and on this machine there is almost none:")
with psycopg.connect("dbname=guide") as conn:
    start = time.perf_counter()
    for _ in range(1000):
        conn.execute("SELECT 1")
    each = (time.perf_counter() - start) / 1000
print(f"  one round trip here is under {max(1, round(each * 1_000_000 / 10) * 10)} microseconds")
print("  over a network it is a thousand times that, and then a pipeline is the whole difference")


5000 statements:
  one at a time: the baseline
  in a pipeline: somewhat faster

what a pipeline removes is the waiting, and on this machine there is almost none:
  one round trip here is under 10 microseconds
  over a network it is a thousand times that, and then a pipeline is the whole difference


This is the failure of a measurement rather than of code. A pipeline is a latency optimization, the
local socket has nearly none, and measuring it here and concluding that pipelines do not help is the
wrong lesson taken from a right number.

Measure it where the program will actually run, or reason about it: statements times latency is the
time a pipeline removes, and nothing else it does matters.


## Recap

- Inside `with conn.pipeline():`, `execute` queues a statement and returns without waiting. The
  results are read afterwards.
- A failure comes back later than the statement that caused it, so it surfaces at whichever result
  is read first, which is usually a statement that was fine.
- One cursor holds one result. A pipeline needs a cursor per statement.
- `pipeline.sync()` is a boundary that finishes everything queued before it.
- `COPY`, named cursors and `stream` are not allowed inside a pipeline, because each needs the
  connection to itself.
- `psycopg.capabilities.has_pipeline()` is the check. It needs libpq 14 or newer, which
  `psycopg[binary]` brings with it.
- A pipeline removes waiting, not work. On a local socket that is almost nothing; over a network it
  is the number of statements times the latency.
- `executemany` already uses pipeline mode, and `COPY` beats both for bulk loading.


## What is next

The **AsyncConnection** notebook is psycopg with `await` in front of it: the event loop a notebook
already has, the coroutine nobody awaited, and the two tasks that share one connection and take
turns rather than going faster.


---

&#8592; **Previous:** [COPY](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/08-copy.ipynb)  &nbsp;·&nbsp;  [asyncpg and psycopg3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)
